# PATH MANAGEMENT

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    batch_size=8, # NOTE: CHANGE    
    max_tok_length=64, # NOTE: CHANGE
    max_epoch=10, # NOTE: CHANGE

    model_name="google/mt5-large"
)

# Fine-tuning

Fine-tuning refers to the process in transfer learning in which the parameter values of a model trained on a large dataset are modified when the training process continues on a small dataset (see [Kevin Murphy's book](https://probml.github.io/pml-book/book1.html) Section 19.2 for further details). The main motivation is to adapt a pre-trained model trained on a large amount of data to tackle a specific task providing better performance that would be achieved training on the small task-specific dataset.

In this notebook, we are going to use for fine-tuning a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to fine-tune the [NLLB model](https://huggingface.co/docs/transformers/model_doc/nllb) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

In [3]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 200965
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [4]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [5]:
raw_datasets["train"][:14]["source_text"]

['artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 'en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.',
 'hubo unos 200 invitados.',
 '¿eres tú mayor de edad?',
 '-"pero no tienes dinero, ¿verdad?"',
 'así que, cuando ella te deja, ¿de donde crees que ella va a hacer a continuación.',
 'para empezar, como ya hemos dicho, debemos tomar la fruta con el estómago vacío.',
 'soy una viuda con cuatro hijos y me quedé atrapado en una situación financiera desde abril de 2016 y necesitaba refinanciar y pagar mis cuentas.',
 'el trabajo con espacios en blanco debe comenzar a fines de la primavera o principios del verano y no retrasarse hasta el otoño para evitar problemas e interrupciones.',
 'es la riqueza guardada por su dueño para su propia desgracia.',
 'buscamos una canción que trate sobre alguno de los siguientes temas: «desarrollo global» o «un solo

In [6]:

raw_datasets["train"][:14]["dest_text"]

['estis mistero por la polico : kial ŝteli nur unu ŝuon anstataù paro ?',
 'la tria jarcento vidis la aperon de kelkaj grandaj okcident ĝermanaj triboj: la alemanoj, frankoj, bavarii-, ĥatoj, saksoj, frisii, sicambri, kaj thuringii.',
 'venis ĉirkaŭ 200 gastoj.',
 'ĉu vi estas la plej aĝa?',
 '"sed vi ne posedas tiom da mono, ĉu ne?"',
 'do, kiam ŝi lasas vin, kie vi kredas, ke ŝi faros poste.',
 'kiel antaŭe menciite, la drogo devas esti prenita sur malplena stomako.',
 'en ĉi tiu tempo mi estas vidvino kun kvar infanoj kaj mi estis ligita en financa situacio en majo 2018 kaj bezonis refinanci kaj pagi miajn biletojn.',
 'laboro kun spacoj devas komenciĝi fine de printempo aŭ frua somero kaj ne malhelpu ĝis aŭtuno por eviti problemojn kaj interrompojn.',
 'riĉecon konservatan por la malutilo de ĝia propra mastro.',
 'tie ĉi mi menciu nur unu temaron, tiun de tutmondiĝo aŭ „globaliĝo”.',
 'ni serĉu rekte la titolon «orientaj tapiŝoj»!',
 'tamen, ĉu tranĉeoj estas por ke ni koncentriĝu'

In [7]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

Provided that the NLLB model was pretrained on sentence pairs involving 200 languages, being one of the them the translation from English into Spanish, we are going to be filtering Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [8]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

Now we load the pre-trained tokenizer for the NLLB model and apply it to the English-Spanish pair:

In [9]:
from transformers import AutoTokenizer

checkpoint = CONFIG.model_name
# mT5 doesn't use src_lang/tgt_lang parameters like NLLB
# It uses task prefixes instead
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    truncation=True, 
    max_length=CONFIG.max_tok_length,
    )

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece 

We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be finetuned:

In [10]:
def preprocess_function(sample):
    # mT5 requires task prefix prepended to source text
    prefixed_sources = [CONFIG.task_prefix + text for text in sample["source_text"]]
    
    model_inputs = tokenizer(
        prefixed_sources, 
        text_target = sample["dest_text"],

        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [11]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[37194, 702, 259, 29037, 288, 26609, 267, 26758, 10491, 569, 269, 53015, 335, 100088, 267, 4228, 7768, 259, 6162, 21113, 259, 130054, 3415, 269, 288, 7198, 268, 291, 1], [37194, 702, 259, 29037, 288, 26609, 267, 289, 362, 2002, 878, 90965, 115487, 12599, 335, 259, 10940, 269, 259, 173234, 106912, 84479, 263, 426, 259, 102984, 259, 12447, 267, 1281, 125018, 261, 75093, 263, 261, 10292, 337, 261, 327, 115201, 261, 71831, 266, 266, 261, 395, 11930, 52224, 261, 259, 276, 2130, 112134, 266, 259, 260, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[259, 5337, 86117, 268, 519, 283, 24774, 268, 259, 267, 504, 473, 39193, 30826, 2593, 18308, 39193, 273, 444, 461, 17676, 1541, 78121, 259, 291, 1], [283, 171071, 26361, 33346, 41469, 26

In [12]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['▁translate', '▁from', '▁', 'Spanish', '▁to', '▁Esperanto', ':', '▁artículo', '▁anterior', 'se', '▁de', 'vela', '▁un', '▁misterio', ':', '▁¿', 'por', '▁', 'qué', '▁moni', '▁', 'argento', '▁era', '▁de', '▁to', 'stad', 'o', '?', '</s>']
['▁translate', '▁from', '▁', 'Spanish', '▁to', '▁Esperanto', ':', '▁en', '▁el', '▁sig', 'lo', '▁iii', '▁surg', 'ieron', '▁un', '▁', 'número', '▁de', '▁', 'tribus', '▁germ', 'ánica', 's', '▁del', '▁', 'oeste', '▁', 'grandes', ':', '▁ale', 'manni', ',', '▁franco', 's', ',', '▁cat', 'os', ',', '▁sa', 'jones', ',', '▁fris', 'i', 'i', ',', '▁si', 'cam', 'bri', ',', '▁', 'y', '▁thu', 'ringi', 'i', '▁', '.', '</s>']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [13]:
tokenizer.batch_decode(model_input['input_ids'])

['translate from Spanish to Esperanto: artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?</s>',
 'translate from Spanish to Esperanto: en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, sajones, frisii, sicambri, y thuringii .</s>']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [14]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map: 100%|██████████| 43063/43063 [00:01<00:00, 30819.46 examples/s]


We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [15]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 64 tokens:   0%|          | 0/200965 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 64 tokens: 100%|██████████| 200965/200965 [00:03<00:00, 57748.25 examples/s]
Discarding source and target sentences with more than 64 tokens: 100%|██████████| 200965/200965 [00:03<00:00, 57748.25 examples/s]
Discarding source and target sentences with more than 64 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 63172.82 examples/s]
Discarding source and target sentences with more than 64 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 63172.82 examples/s]
Discarding source and target sentences with more than 64 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 63994.33 examples/s]
Discarding source and target sentences with more than 64 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 63994.33 examples/s]


We can take a quick look at the length histogram in the source language:

In [16]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

10  26
11 294
12 1351
13 3611
14 6555
15 9415
16 10892
17 11394
18 11457
19 10300
20 9349
21 8501
22 7490
23 6448
24 5658
25 5076
26 4568
27 4071
28 3806
29 3511
30 3223
31 3017
32 2850
33 2615
34 2594
35 2446
36 2395
37 2284
38 2231
39 2080
40 2074
41 1922
42 1854
43 1793
44 1811
45 1774
46 1695
47 1667
48 1636
49 1532
50 1454
51 1464
52 1318
53 1370
54 1286
55 1194
56 1166
57 1103
58 1029
59 1047
60 982
61 922
62 866
63 806
64 808


Checking a sample after filtering by maximum number of tokens:

In [17]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[37194, 702, 259, 29037, 288, 26609, 267, 26758, 10491, 569, 269, 53015, 335, 100088, 267, 4228, 7768, 259, 6162, 21113, 259, 130054, 3415, 269, 288, 7198, 268, 291, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[259, 5337, 86117, 268, 519, 283, 24774, 268, 259, 267, 504, 473, 39193, 30826, 2593, 18308, 39193, 273, 444, 461, 17676, 1541, 78121, 259, 291, 1]
[37194, 702, 259, 29037, 288, 26609, 267, 289, 362, 2002, 878, 90965, 115487, 12599, 335, 259, 10940, 269, 259, 173234, 106912, 84479, 263, 426, 259, 102984, 259, 12447, 267, 1281, 125018, 261, 75093, 263, 261, 10292, 337, 261, 327, 115201, 261, 71831, 266, 266, 261, 395, 11930, 52224, 261, 259, 276, 2130, 112134, 266, 259, 260, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[283, 171071, 26361, 33346, 41469, 263, 283, 103414, 444, 269, 8650, 6718, 6130, 1030, 25

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [18]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [19]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )


Next, you should call the prepare_model_for_kbit_training() function to preprocess the quantized model for training.

In [20]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False, gradient_checkpointing_kwargs={'use_reentrant':False})

[LoRA (Low-Rank Adaptation of Large Language Models)](https://huggingface.co/docs/peft/task_guides/lora_based_methods) is a [parameter-efficient fine-tuning (PEFT)](https://huggingface.co/docs/peft/index) technique that significantly reduces the number of trainable parameters. It works by inserting a smaller number of new weights into the model and only these are trained. This makes training with LoRA much faster, memory-efficient, and produces smaller model weights (a few hundred MBs), which are easier to store and share.

Each PEFT method is defined by a PeftConfig class that stores all the important parameters for building a PeftModel. For example, to train with LoRA, load and create a LoraConfig class and specify the following parameters:

<ul>
<li>task_type: the task to train for (sequence-to-sequence language modeling in this case)</li>
<li>r: the dimension of the low-rank matrices</li>
<li>lora_alpha: the scaling factor for the low-rank matrices</li>
<li>target_modules: determine what set of parameters are adapted</li>
<li>lora_dropout: the dropout probability of the LoRA layers</li>
</ul>

In [21]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=32, # NOTE: CHANGE
    lora_alpha=64, # NOTE: CHANGE
    # mT5 uses different module names than NLLB
    # For mT5, attention modules are named: q, k, v, o (same as T5)
    target_modules=["q", "k", "v", "o"],
    lora_dropout=0.1, # NOTE: CHANGE
    bias="none",
)

Once LoRA and the quantization are setup, create a quantized PeftModel with the get_peft_model() function. It takes a quantized model and the LoraConfig containing the parameters for how to configure a model for training with LoRA.

In [22]:
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 18,874,368 || all params: 1,248,455,680 || trainable%: 1.5118


The function that is responsible for putting together samples inside a batch is called a collate function. It is an argument you can pass when you build a DataLoader, the default being a function that will just convert your samples to PyTorch tensors and concatenate them. This is not possible in our case since the inputs we have are not all of the same size. We have deliberately postponed the padding, to only apply it as necessary on each batch and avoid having over-long inputs with a lot of padding.

To do this in practice, we have to define a collate function that will apply the correct amount of padding to the items of the dataset we want to batch together. Fortunately, the Transformers library provides us with such a function via DataCollatorForSeq2Seq that takes a tokenizer when you instantiate it (to know which padding token to use, and whether the model expects padding to be on the left or on the right of the inputs), so we will also need to instantiate the model first to provide it to the collate function:

In [23]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, 
    model=lora_model, 
    pad_to_multiple_of=8
    )

## Evaluation

The last thing to define for our Seq2SeqTrainer is how to compute the metrics to evaluate the predictions of our model with respect to references. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu). You can see a simple example of usage below:

:

In [24]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")
metric_chrf = load("chrf")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 122640.47it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `

We need to define a function compute_metrics to compute BLEU scores at each epoch. The example below performs a basic post-processing to decode the predictions into texts:

In [25]:
from torch.nn.utils.rnn import pad_sequence
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics_train(eval_preds):
    preds, labels = eval_preds

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # NOTE: CHANGE 
    # Replace invalid token IDs in predictions to prevent OverflowError
    # Clip to valid range and handle any overflow issues
    vocab_size = len(tokenizer)
    preds = np.array(preds)
    preds = np.where((preds < 0) | (preds >= vocab_size) | np.isnan(preds) | np.isinf(preds), 
                     tokenizer.pad_token_id, 
                     preds)
    preds = np.clip(preds, 0, vocab_size - 1).astype(np.int64)
    preds = preds.tolist()
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_chrf = metric_chrf.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result = {
        "chrf": result_chrf["score"],
    }
    

    prediction_lens = [np.count_nonzero(np.array(pred) != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


def compute_metrics(preds, labels, sources):

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # NOTE: CHANGE 
    # Replace invalid token IDs in predictions to prevent OverflowError
    if hasattr(preds, '__iter__') and len(preds) > 0:
        # Check if preds contains tensors
        if hasattr(preds[0], 'tolist'):
            # Pad sequences to same length
            preds_padded = pad_sequence(
                [p if len(p.shape) > 0 else p.unsqueeze(0) for p in preds], 
                batch_first=True,   
                padding_value=tokenizer.pad_token_id
            )
            preds = preds_padded.cpu().numpy()
        else:
            preds = np.array(preds) if not isinstance(preds, np.ndarray) else preds
             
    # Clip to valid range and handle any overflow issues
    vocab_size = len(tokenizer)
    # preds = np.array(preds)
    preds = np.where((preds < 0) | (preds >= vocab_size) | np.isnan(preds) | np.isinf(preds), 
                     tokenizer.pad_token_id, 
                     preds)
    preds = np.clip(preds, 0, vocab_size - 1).astype(np.int64)
    preds = preds.tolist()
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Decode sources
    # decoded_sources = tokenizer.batch_decode(sources, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_blue = metric_bleu.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result_comet = metric_comet.compute(
        sources=sources,
        predictions=decoded_preds, 
        references=[label[0] for label in decoded_labels]  # COMET expects flat list, not nested
    )
    result_chrf = metric_chrf.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result = {
        "bleu": result_blue["score"],
        "comet": result_comet["mean_score"],
        "chrf": result_chrf["score"]
    }

    prediction_lens = [np.count_nonzero(np.array(pred) != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Training

The first step before we can define our [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer#trainer) is to define a [Seq2SeqTrainingArguments class](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Seq2SeqTrainingArguments) that will contain all the hyperparameters the Trainer will use for training and evaluation. The only compulsory argument you have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way. For all the rest, you can set them depending on the recommendations from the model developers:

In [26]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    CONFIG.output_model_dir,
    eval_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=CONFIG.batch_size,
    per_device_eval_batch_size=CONFIG.batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=CONFIG.max_epoch,
    predict_with_generate=True,

    generation_max_length=CONFIG.max_tok_length,
    generation_num_beams=5, # NOTE: CHANGE
    metric_for_best_model="chrf",   # NOTE: CHANGE # Since you're using chrF
)

Once we have our model, we can define a Trainer by passing it all the objects constructed up to now — the model, the training_args, the training and validation datasets, the tokenizer, the data collator and the compute_metrics function:

In [27]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    lora_model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['valid'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_train
)


/tmp/ipykernel_1754667/1397975236.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


To fine-tune the model on our dataset, we just have to call the [train() function](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.train) of our Trainer:

In [28]:
# trainer.train()

The training stop because pc got powered down... need to continue training jeje

The model seems to start overfitting when taking a look at the validations, we gona let it finish but me end up with worse results

In [29]:
# import os
# import glob

# # Find the latest checkpoint in the output directory
# checkpoint_dirs = glob.glob(os.path.join(CONFIG.output_model_dir, "checkpoint-*"))
# if checkpoint_dirs:
#     latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))
#     print(f"Resuming training from checkpoint: {latest_checkpoint}")
    
#     trainer.train(resume_from_checkpoint=latest_checkpoint)
# else:
#     print("No checkpoints found. Starting training from scratch.")
#     trainer.train()

In [30]:
# # Save the final model
# trainer.save_model(os.path.join(CONFIG.output_model_dir, "final_model"))

In [31]:
# Load the final model
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel

# Load the base model with quantization
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
)

# Load the LoRA adapter weights
model = PeftModel.from_pretrained(
    base_model,
    os.path.join(CONFIG.output_model_dir, "final_model")
)

## Inference

At inference time, it is recommended to use [generate()](https://huggingface.co/docs/transformers/v4.26.1/en/main_classes/text_generation#transformers.GenerationMixin.generate). This method takes care of encoding the input and feeding the encoded hidden states via cross-attention layers to the decoder and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers. There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB: 

In [32]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0
}



We prepare the test set in batches to be translated:

In [33]:
batch_tokenized_test = tokenized_datasets['test'].batch(CONFIG.batch_size)

Batching examples: 100%|██████████| 39463/39463 [00:02<00:00, 15591.04 examples/s]


Processing in batches to add padding and converting to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

In [34]:
config_greedy = {
    "name": "greedy",   
    "num_beams": 1,
    "do_sample": False, 
    "temperature": 0,  
    "top_k": 0,
    "top_p": 1.0,
}

config_beam = {
    "name": "beam",
    "num_beams": 5,
    "do_sample": False,
    "temperature": 0,  
    "top_k": 0,
    "top_p": 1.0,
}

config_sampling = {
    "name": "sampling",
    "num_beams": 3,
    "do_sample": True, # Stochastic predictions
    "temperature": 0.2,  
    "top_k": 30,
    "top_p": 0.90,
}

config_random = {
    "name": "random",
    "num_beams": 3,
    "do_sample": True,
    "temperature": 0.6,  
    "top_k": 30,
    "top_p": 0.90,
}

tests = [
    config_greedy,
    config_beam,
    config_sampling,
    config_random
]

In [35]:
import tqdm
from maikol_utils.print_utils import print_separator

number_of_batches = len(batch_tokenized_test["source_text"])
all_sources = {}
output_sequences = {}
for test in tests:
    all_sources[test['name']] = []
    output_sequences[test['name']] = []
    print_separator(test['name'])

    for i in tqdm.tqdm(range(number_of_batches)):
        all_sources[test['name']].extend(batch_tokenized_test["source_text"][i])

        inputs = tokenizer(
            batch_tokenized_test["source_text"][i], 
            max_length=CONFIG.max_tok_length, 
            truncation=True, 
            return_tensors="pt", 
            padding=True,
            )
        with torch.no_grad():    
            output_batch = model.generate(
                generation_config=generation_config, 
                input_ids=inputs["input_ids"].cuda(), 
                attention_mask=inputs["attention_mask"].cuda(), 
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), 
                max_length = CONFIG.max_tok_length, 
                num_beams=test['num_beams'], 
                do_sample=test['do_sample'],
                temperature=test['temperature'] if test['do_sample'] else None,
                top_k=test['top_k'] if test['do_sample'] else None,
                top_p=test['top_p'] if test['do_sample'] else None,
                )
        output_sequences[test['name']].extend(output_batch.cpu())

________________________________________________________________
                             greedy                             



100%|██████████| 4933/4933 [1:44:06<00:00,  1.27s/it]


________________________________________________________________
                              beam                              



100%|██████████| 4933/4933 [2:09:54<00:00,  1.58s/it]


________________________________________________________________
                            sampling                            



100%|██████████| 4933/4933 [2:27:36<00:00,  1.80s/it]


________________________________________________________________
                             random                             



100%|██████████| 4933/4933 [2:12:11<00:00,  1.61s/it]


In [36]:
all_sources

{'greedy': ['“la cual deja al compañero de su juventud, y olvida el pacto de su dios”;',
  'durante la manía, un individuo se comporta o se siente anormalmente enérgico, feliz, o irritable.',
  'abro los ojos, deslumbrado.',
  'ellos preferirán la cremación.',
  'alexander hamilton comenzó la "u. s. treasury" con nada, y eso fue lo más cerca que ha estado nuestro país nunca de llegar a ser justo.',
  'el enemigo de mi enemigo es mi amigo, reza el dicho.',
  'el servicio de restaurante es sólo para grupos.',
  'jamás lo olvidaré, y estoy absolutamente agradecida por todo eso.',
  'garantizo sin embargo que será emocionalmente intensa.',
  'ellos usan sus plumas naturalmente mudadas para ceremonias religiosas y culturales.',
  'consejos de hvac',
  'una camiseta exclusiva como esta:',
  'después burgués dejó el cargo para unirse a un convento en 1944, matisse veces en contacto con ella para solicitar que modelar para él.',
  'podrían manchar tu piel.',
  'dos dedos en cada mano',
  'el s

In [37]:
results = {}
for test in tests:
    print_separator(test['name'])
    result = compute_metrics(
        output_sequences[test['name']], 
        tokenized_datasets["test"]["labels"], 
        all_sources[test['name']]
    )
    results[test['name']] = result
    print(f'BLEU score: {result["bleu"]:0.4f}')
    print(f'COMET score: {result["comet"]:0.4f}')
    print(f'CHRF score: {result["chrf"]:0.4f}')

________________________________________________________________
                             greedy                             



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You a

BLEU score: 23.2478
COMET score: 0.7459
CHRF score: 50.2323
________________________________________________________________
                              beam                              



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the f

BLEU score: 23.5096
COMET score: 0.7604
CHRF score: 52.2605
________________________________________________________________
                            sampling                            



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the f

BLEU score: 23.8724
COMET score: 0.7599
CHRF score: 52.0065
________________________________________________________________
                             random                             



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the f

BLEU score: 23.7808
COMET score: 0.7594
CHRF score: 52.0821


In [38]:
# Plot the results for BLEU COMET and CHRF for each decoding strategy
import plotly.graph_objects as go

# Extract data
strategies = list(results.keys())
bleu_scores = [results[s]['bleu'] for s in strategies]
comet_scores = [results[s]['comet'] * 100 for s in strategies]  # Scale COMET to 0-100
chrf_scores = [results[s]['chrf'] for s in strategies]

# Create figure with grouped bars
fig = go.Figure()

# Add BLEU bars
fig.add_trace(go.Bar(
    name='BLEU',
    x=strategies,
    y=bleu_scores,
    text=[f'{score:.2f}' for score in bleu_scores],
    textposition='outside',
    marker_color='#3498db'
))

# Add COMET bars (scaled x100)
fig.add_trace(go.Bar(
    name='COMET (x100)',
    x=strategies,
    y=comet_scores,
    text=[f'{score:.2f}' for score in comet_scores],
    textposition='outside',
    marker_color='#2ecc71'
))

# Add CHRF bars
fig.add_trace(go.Bar(
    name='CHRF',
    x=strategies,
    y=chrf_scores,
    text=[f'{score:.2f}' for score in chrf_scores],
    textposition='outside',
    marker_color='#f39c12'
))

# Update layout
fig.update_layout(
    title='Translation Quality Metrics by Decoding Strategy',
    title_font_size=16,
    xaxis_title='Decoding Strategy',
    yaxis_title='Score (0-100)',
    barmode='group',
    yaxis=dict(range=[0, 100]),
    height=600,
    width=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

In [39]:
best = max([(name, scores['chrf']) for name, scores in results.items()], key=lambda x: x[1])

In [40]:
from maikol_utils.print_utils import print_separator

# all_sources already contains raw text strings
# output_sequences contains token IDs that need to be decoded
decoded_outputs = tokenizer.batch_decode(output_sequences[best[0]], skip_special_tokens=True)

# Get reference translations from test set
test_references = tokenized_datasets["test"]["dest_text"]

# Print first 10 examples
for i, (source, output, reference) in enumerate(zip(all_sources[best[0]][:10], decoded_outputs[:10], test_references[:10])):
    print_separator(f"Example {i+1}:")
    print(f"Source:      {source}")
    print(f"Translation: {output}")
    print(f"Reference:   {reference}")

________________________________________________________________
                           Example 1:                           

Source:      “la cual deja al compañero de su juventud, y olvida el pacto de su dios”;
Translation: n, kiu forlasas la amikon de sia juneco, kaj forgesas la interkonsenton de sia dio;
Reference:   kiu forlasas la amikon de sia juneco, kaj forgesas la ligon de sia dio;
________________________________________________________________
                           Example 2:                           

Source:      durante la manía, un individuo se comporta o se siente anormalmente enérgico, feliz, o irritable.
Translation: dum la manĝo, individuo kondutas aŭ sentiĝas nenormale energia, feliĉa, aŭ irrita.
Reference:   [2] dum manio, individuo kondutas aŭ sentiĝas nenormale energia, feliĉa, aŭ agaciĝema.
________________________________________________________________
                           Example 3:                           

Source:      abro los ojos, des